# Train Faster R-CNN ResNet-50-FPN trên Kaggle

Notebook này là giao diện chạy online cho pipeline trong repository. Bật **GPU** và **Internet** trong Kaggle Settings, add dữ liệu VisDrone vào Notebook Input, sửa cell cấu hình rồi chọn **Run All**.

- `F0`: baseline pretrained COCO, không augmentation.
- `F1`: giữ nguyên cấu hình F0, chỉ thêm augmentation bbox-safe.
- Smoke test chạy 2 batch để kiểm tra pipeline; kết quả không dùng để báo cáo.


## 1. Cấu hình

Nếu repository hoặc dataset không được tự động tìm thấy, điền đường dẫn Kaggle Input tương ứng. Khi resume từ một Kaggle Output cũ, đặt `RESUME_FROM` tới file `last.pth` đã add làm Input.


In [ ]:
from pathlib import Path

EXPERIMENT = "F0"            # F0 hoặc F1
SMOKE_TEST = True             # chạy True trước; đổi False để train đủ 25 epoch
EPOCHS = 25
BATCH_SIZE = 2
WORKERS = 2
SEED = 42

PROJECT_ROOT_OVERRIDE = None  # ví dụ: /kaggle/input/object-detection-repo/Object_dectection
RAW_DATA_ROOT = None          # ví dụ: /kaggle/input/visdrone-dataset
PROCESSED_DATA_ROOT = None    # dùng nếu Input đã có images/ và annotations/instances_train.json
RESUME_FROM = None            # ví dụ: /kaggle/input/f0-checkpoint/f0/last.pth
OUTPUT_ROOT = Path("/kaggle/working/faster_rcnn_runs")

assert EXPERIMENT in {"F0", "F1"}


## 2. Tìm repository và cài dependency


In [ ]:
import importlib.util
import subprocess
import sys

def find_project_root():
    if PROJECT_ROOT_OVERRIDE:
        candidate = Path(PROJECT_ROOT_OVERRIDE)
        if (candidate / "src" / "faster_rcnn.py").is_file():
            return candidate.resolve()
        raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE không hợp lệ: {candidate}")
    direct = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working")]
    for candidate in direct:
        if (candidate / "src" / "faster_rcnn.py").is_file():
            return candidate.resolve()
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = list(base.glob("**/src/faster_rcnn.py"))
            if matches:
                return matches[0].parents[1].resolve()
    raise FileNotFoundError("Không tìm thấy repository; hãy đặt PROJECT_ROOT_OVERRIDE.")

PROJECT_ROOT = find_project_root()
if importlib.util.find_spec("pycocotools") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools>=2.0.7"])

import torch, torchvision
assert torch.cuda.is_available(), "Chưa có CUDA. Vào Kaggle Settings và bật GPU Accelerator."
print({"project": str(PROJECT_ROOT), "torch": torch.__version__,
       "torchvision": torchvision.__version__, "gpu": torch.cuda.get_device_name(0)})


## 3. Chuẩn bị COCO dataset

Nếu Input đã chứa bản processed thì notebook dùng trực tiếp. Nếu chỉ có dữ liệu VisDrone gốc, notebook tạo COCO JSON và symlink ảnh vào `/kaggle/working` để tránh copy thêm vài GB.


In [ ]:
def valid_processed_root(path):
    path = Path(path)
    return all((path / item).exists() for item in (
        "images", "annotations/instances_train.json", "annotations/instances_val.json"
    ))

if PROCESSED_DATA_ROOT:
    DATA_ROOT = Path(PROCESSED_DATA_ROOT)
    if not valid_processed_root(DATA_ROOT):
        raise FileNotFoundError(f"Processed dataset không hợp lệ: {DATA_ROOT}")
else:
    candidates = []
    for base in (Path("/kaggle/input"), PROJECT_ROOT / "data"):
        if base.exists():
            candidates.extend(p.parent.parent for p in base.glob("**/annotations/instances_train.json"))
    DATA_ROOT = next((p for p in candidates if valid_processed_root(p)), None)

if DATA_ROOT is None:
    raw_root = Path(RAW_DATA_ROOT) if RAW_DATA_ROOT else Path("/kaggle/input")
    DATA_ROOT = Path("/kaggle/working/visdrone_processed")
    command = [
        sys.executable, str(PROJECT_ROOT / "src" / "preprocess_visdrone.py"),
        "--data-root", str(raw_root), "--output-root", str(DATA_ROOT),
        "--splits", "train", "val", "--image-mode", "symlink",
    ]
    subprocess.check_call(command, cwd=PROJECT_ROOT)

assert valid_processed_root(DATA_ROOT)
print("Processed dataset:", DATA_ROOT)


## 4. Train, evaluate và lưu checkpoint

Lần đầu giữ `SMOKE_TEST=True`. Sau khi thành công, đổi thành `False`, restart session rồi Run All. Output gồm `last.pth`, `best.pth`, `history.csv`, `learning_curves.png`, `config.json` và `summary.json`.


In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "train_faster_rcnn.py"),
    "--data-root", str(DATA_ROOT),
    "--output-dir", str(OUTPUT_ROOT),
    "--experiment", EXPERIMENT,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--workers", str(WORKERS),
    "--seed", str(SEED),
]
if SMOKE_TEST:
    command.append("--smoke-test")
if RESUME_FROM:
    command.extend(["--resume", str(RESUME_FROM)])

print("Running:", " ".join(command))
subprocess.check_call(command, cwd=PROJECT_ROOT)


## 5. Xem kết quả và tạo Kaggle Output


In [ ]:
import json
import pandas as pd
from IPython.display import Image as DisplayImage, display

RUN_DIR = OUTPUT_ROOT / EXPERIMENT.lower()
summary = json.loads((RUN_DIR / "summary.json").read_text())
display(pd.DataFrame([summary]))
display(pd.read_csv(RUN_DIR / "history.csv"))
if (RUN_DIR / "learning_curves.png").is_file():
    display(DisplayImage(filename=str(RUN_DIR / "learning_curves.png")))
print("Artifacts để Save Version / tải về:", RUN_DIR)
